# Dependencies

In [47]:
from langchain.document_loaders import PyPDFDirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from pinecone import Pinecone, ServerlessSpec
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate 
from dotenv import load_dotenv
load_dotenv()
import os
import time

In [48]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_API_ENV = os.getenv("PINECONE_API_ENV")

# Loading and chunking the Data
Here, we will use the PyPDFDirectoryLoader for the PDF from the langchain wrapper to load the PDF data and chunk it into paragraphs. First step includes loading the current PDF file into the loader and converting it to a list of documents. Each document is a list of pages.

In [49]:
loader = PyPDFDirectoryLoader("../data")
data = loader.load()
data

[Document(metadata={'source': '../data/Plan_India_Knowledge_Base.pdf', 'page': 0}, page_content='VolunteeringPolicy\n[A]SCOPEANDPURPOSE\nThepurposeofvolunteeringistoofferanincredibleopportunitytoindividualstocontributetowardsthebettermentofsociety.Avolunteercangivetimeandskillsandexperiencepersonalandprofessional growth.\nVolunteerswill beattheheartofthePlanIndiadevelopmentactionfutureandarevital tooursuccess.Wegainenormousvaluefromvolunteers,benefittingfromavarietyofexperienceandtalentstodeliverourprogramsmoreeffectively.PlanIndiainvolvesvolunteersbecausethey–1. Offerapool ofskillsandexperiencewewouldnototherwisehaveaccessto.2. Bringextracredibilitytoourwork–volunteerschooseusanddonatetheirtimeandtalentsfreely.3. Championourmissionandextendourreachincommunities.4. Bringfreshperspective–volunteersarenotfinanciallydependentontheorganisationsocanbebetterplacedthanstafftochallengeus,guideus,andbringnewideas.5. EnhancethespiritofPlanIndiawithenthusiasm,passionandcommitment.\n[B]LEGALSTATUS

Now, since the context window of our LLM will be limited, the ideal way to handle this is to chunk the data into paragraphs. This is done by the chunker, which takes the list of documents and returns a list of paragraphs within the chunk limit we set (100 words in this case). The chunker also takes care of the page breaks and ensures that the paragraphs are not split across pages. Also, we will be introducing an overlap, which will be the number of words that will be repeated in the end of one chunk and the beginning of the next chunk. This is done to ensure that the context is not lost between the chunks.

In [50]:
text_split = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
text_chunks = text_split.split_documents(data)

text_chunks

[Document(metadata={'source': '../data/Plan_India_Knowledge_Base.pdf', 'page': 0}, page_content='VolunteeringPolicy\n[A]SCOPEANDPURPOSE\nThepurposeofvolunteeringistoofferanincredibleopportunitytoindividualstocontributetowardsthebettermentofsociety.Avolunteercangivetimeandskillsandexperiencepersonalandprofessional growth.'),
 Document(metadata={'source': '../data/Plan_India_Knowledge_Base.pdf', 'page': 0}, page_content='Volunteerswill beattheheartofthePlanIndiadevelopmentactionfutureandarevital tooursuccess.Wegainenormousvaluefromvolunteers,benefittingfromavarietyofexperienceandtalentstodeliverourprogramsmoreeffectively.PlanIndiainvolvesvolunteersbecausethey–1. Offerapool ofskillsandexperiencewewouldnototherwisehaveaccessto.2. Bringextracredibilitytoourwork–volunteerschooseusanddonatetheirtimeandtalentsfreely.3. Championourmissionandextendourreachincommunities.4.'),
 Document(metadata={'source': '../data/Plan_India_Knowledge_Base.pdf', 'page': 0}, page_content='Bringfreshperspective–vol

In [51]:
print(f"Length of chunks : {len(text_chunks)}")

Length of chunks : 50


# Pinecone Initialization
Now, we will be using the pinecone vectorDB to store the embeddings of the chunks. We will be using the `pinecone.init()` function to initialize the pinecone environment. We will be using the `pinecone.use_index()` function to use the index created for this project and setup the instance for the same.

In [52]:
pc = Pinecone(api_key = PINECONE_API_KEY, environment = PINECONE_API_ENV)

Now, let us view the indexs avaliable in the pinecone environment.

In [53]:
pc.list_indexes()

[
    {
        "name": "knowledge-base",
        "dimension": 1536,
        "metric": "cosine",
        "host": "knowledge-base-f1hj4e6.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "deletion_protection": "disabled"
    },
    {
        "name": "documents",
        "dimension": 1536,
        "metric": "cosine",
        "host": "documents-f1hj4e6.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "deletion_protection": "disabled"
    }
]

Here, we will be using the knowledge base index to store the embeddings of the chunks. We will be using the `pinecone.use_index()` function to use the index created for this project and setup the instance for the same.

# Embedding the Chunks using OpenAI text-embedding-3-small
Here, we will be using the OpenAI text-embedding-3-small model to embed the chunks, for which we will need an openAI instance initialised.

In [54]:
openAI_client = OpenAI(api_key=OPENAI_API_KEY)

Let us go ahead and set the embeddings model and a function to get the embeddings of any given text via the text-embedding-3-small model.

In [55]:
embedding_model = openAI_client.embeddings

def get_embedding(text) :
    response = embedding_model.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

Now, each entry in the vectorDB should have : 

* **ID** : The unique ID of the document, which will be a combination of the page number and chunk number.
* **VALUES** : The embedding of the chunk, as generated by the OpenAI text-embedding-3-small model.
* **METADATA** : The metadata of the document, which will include the chunk information.
    *  **CHUNK** : The text of the chunk.
    *  **PAGE_NUMBER** : The page number of the chunk in the document.
    *  **CHUNK_INDEX** : The number of the chunk in the document.

In [56]:
def create_vectors(text_chunks):
    v = []
    chunk_num = 0
    
    for chunk in text_chunks: 
        page_num = chunk.metadata["page"]
        
        entry = {}
        entry["id"] = f"PAGE_{page_num}_CHUNK_{chunk_num}"
        entry["values"] = get_embedding(chunk.page_content)
        entry["metadata"] = {
            "chunk" : chunk.page_content,
            "page_number" : chunk.metadata["page"],
            "chunk_number" : chunk_num
        }
        
        chunk_num += 1
        v.append(entry)
        
    return v

In [57]:
vectors = create_vectors(text_chunks)

With this, we have our vectors stored in the ideal format to be pushed into the vector DB. Let us now push the vectors into the vectorDB of pinecone.

# Pushing the Vectors into the Pinecone Index

In [58]:
index_name = "knowledge-base"
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

index = pc.Index(index_name)

index.upsert(
    vectors=vectors
)

{'upserted_count': 50}

# Querying the Vectors
We shall now query the vectors to check if the vectors have been stored correctly in the pinecone index, and how does this exactly work. We will fetch the relevant vectors from the pinecone index. For that, we will create a function which takes a text query, converts into to an embedding and queries the pinecone index to get the most similar texts from the vectors stored in the index.

In [59]:
def get_relevant_chunks(query):
    query_vector = get_embedding(query)
    
    results = index.query(
        vector = query_vector,
        top_k = 3,
        include_values = False,
        include_metadata = True,
    )
    
    relevant_texts = []
    for record in results['matches']:
        text = {}
        text['score'] = record['score']
        text['text'] = record['metadata']['chunk']
        text["reference"] = int(record["metadata"]["page_number"]) + 1
        relevant_texts.append(text)
    
    return relevant_texts

Finally, we can create a QA system which will take a query and return the most relevant chunks from the PDF document.

In [60]:
import sys 
while True:
    user_input = input(f"Input Prompt: ")
    if user_input=='exit':
        print( 'Exiting')
        sys.exit()
    if user_input == '':
        continue
    
    docs = get_relevant_chunks(user_input)
        
    for doc in docs:
        print(f"Rank {doc['score']} \n Reference {doc['reference']} \n Answer: \n {doc['text']}")
        print("------------------------")

    print("------------------------------------------------------------------------------------------------------------------------")
        

Rank 0.486355364 
 Reference 6 
 Answer: 
 OURHISTORYThePlanwasfoundedin1937byBritishjournalistJohnLangdon-DaviesandrefugeeworkerEricMuggeridge.Originallynamed‘FosterParentsPlanforChildreninSpain’,theaimwastoprovidefood,accommodation,andeducationtochildrenwhoseliveshadbeendisruptedbytheSpanishCivilWar.
Sincethattime,theirapproachtohumanitarianassistancehasevolvedfromwartimereliefactivitiestopost-warsupport,tolong-termcommunitydevelopmentandemergencyassistancethathelpschildren,theirfamilies,andtheircommunitiesindevelopingcountries.
------------------------
Rank 0.364657909 
 Reference 4 
 Answer: 
 ABOUT
WHOAREWEPlanInternational(IndiaChapter),alsoreferredtoasPlanIndia,isanIndianregisterednot-for-profitorganisationthatisconstantlystrivingtoadvancewelfareanddevelopmentforchildrenandequalityforallgirlsandwomeninIndia.Throughitsgrassrootssocialdevelopmentwork,PlanIndiaseekstocreatelastingimpactinthelivesofpoorandvulnerablechildren,theirfamiliesandcommunities,bygendertransformativechild-cen

SystemExit: 

/Users/dhruv/Desktop/VolunteerBot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


With this, our pipeline is complete and we can now move on to the next steps which is sending these relvant documents to the LLM to answer our query.

# Prompt Template for the LLM
Here, we will need to define the prompt for the LLM to answer the query. The LLM will be given the query and the relevant documents, and it will be expected to return the answer to the query. 

In [61]:
query_prompt_template = """
    You are a specialised volunteer at the Plan India NGO, and you will be assisting the potential users to answer their queries. 
    Your personality is that of a helpful and knowledgeable person, who is always ready to help. Think of yourself as a senior 
    volunteer who is very compassionate for the NGO and wants to help assist the users as much as possible, eventually aiming to 
    make them volunteers as well.
    
    
    You will be given the top relevant documents and you have to use those to answer the query asked by the user, which will be given to you below. 
    In the relevant documents,you will be given the cosine similarity score, the reference (which is the page number where this 
    text was in the document) and the text itself. You can in you answer integrate the reference to build authenticity of your answer, 
    by precisely writing it like (reference page : page_num)
    
    \n\n User Query : {query}
    \n\n Documents : {documents}
    
    MAKE SURE YOU DO NOT ANSWER FROM ANYTHING APART FROM THE DOCUMENTS GIVEN TO YOU. 
"""

In [62]:
query_prompt = PromptTemplate(
    input_variables=["query","documents"],
    template=query_prompt_template
)

# Initializing the LLM Client and Chain for RAG Model

In [63]:
chat = ChatOpenAI(
    temperature = 0,  
    model = "gpt-4o",
    openai_api_key = OPENAI_API_KEY
)

In [64]:
query_chain = LLMChain(
    llm=chat,
    prompt=query_prompt
)

# Q&A System using the chain

In [66]:
user_query = "what is your approach based on gender?"
docs = get_relevant_chunks(user_query)

# Run the chain
response = query_chain.invoke({
    "query": user_query,
    "documents": docs
})

# Print the response
print("Response from LLM:")
print(response['text'])

Response from LLM:
Plan India's approach to gender is based on a gender-transformative framework. This approach goes beyond merely addressing the symptoms of gender inequality. Instead, it explicitly tackles the root causes, such as unequal gender power relations, discriminatory social norms, and systemic issues in structures, policies, and practices. The aim is to improve the daily conditions of girls while also advancing their position and value in society (reference page: 6).

The approach includes several key elements:

1. Understanding and addressing how gender norms influence children throughout their life course.
2. Building girls' agency over decisions that affect them, and enhancing their knowledge, confidence, skills, and access to and control over resources.
3. Working with and supporting boys, young men, and men to promote gender equality.
4. Considering girls, boys, young women, and young men in all their diversity.
5. Improving the conditions and social position of girls 